In [1]:
import json
import numpy as np
from scipy.stats import mannwhitneyu

def mann_whitney_u_test(autofl_ranks, reportfl_ranks):
    """
    Perform Mann-Whitney U test.

    Returns
    -------
    U : float
        Mann-Whitney U statistic
    p_value : float
        p-value of the test
    """

    U, p_value = mannwhitneyu(
        autofl_ranks,
        reportfl_ranks,
        alternative="two-sided"
    )

    return U, p_value

def vargha_delaney_A12(autofl_ranks, reportfl_ranks):
    """
    Compute Vargha-Delaney Â12 effect size.

    A12 > 0.5  -> AutoFL better
    A12 = 0.5  -> no difference
    A12 < 0.5  -> ReportFL better
    """

    n1 = len(autofl_ranks)
    n2 = len(reportfl_ranks)

    U, _ = mannwhitneyu(
        reportfl_ranks,
        autofl_ranks,
        alternative="two-sided"
    )

    A12 = U / (n1 * n2)

    return A12

def extract_best_ranks(data):
    bug_ranks = {}

    for bug, methods in data["buggy_methods"].items():
        ranks = []

        for m in methods.values():
            ranks.append(m["autofl_rank"])

        if ranks:
            bug_ranks[bug] = min(ranks)

    return bug_ranks

def get_ranks(exp_name, gpt_version, repetition):
    score_path_json = f"../combined_fl_results/{exp_name}/{gpt_version}_R{repetition}_full_light.json"
    with open(score_path_json, "r") as f:
        data = json.load(f)
        return extract_best_ranks(data)

autofl_ranks = get_ranks("autofl", "gpt-3.5-turbo-0125", 1)
reportfl_ranks = get_ranks("reportfl", "gpt-3.5-turbo-0125", 1)

common_bugs = sorted(set(autofl_ranks) & set(reportfl_ranks))

autofl_res = [autofl_ranks[b] for b in common_bugs]
reportfl_res = [reportfl_ranks[b] for b in common_bugs]

U, p = mann_whitney_u_test(autofl_res, reportfl_res)
A12 = vargha_delaney_A12(autofl_res, reportfl_res)

print("Mann-Whitney U:", U)
print("p-value:", p)
print("Vargha-Delaney A12:", A12)

Mann-Whitney U: 29995.5
p-value: 2.718253679426185e-09
Vargha-Delaney A12: 0.6383644386574074
